## Import Libraries

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pathlib import Path

## Load Raw Data

In [2]:
RAW = Path("../data/raw") 
orders = pd.read_csv(RAW / "olist_orders_dataset.csv") 
customers = pd.read_csv(RAW / "olist_customers_dataset.csv") 
items = pd.read_csv(RAW / "olist_order_items_dataset.csv") 
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv") 
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv") 
products = pd.read_csv(RAW / "olist_products_dataset.csv") 
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv") 
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv") 
cat_trans = pd.read_csv(RAW / "product_category_name_translation.csv") 
dfs = {"orders":orders, "customers":customers, "items":items, "payments":payments, "reviews":reviews, "products":products, "sellers":sellers, "geo":geo, "cat_trans":cat_trans}

 ## Shape & dtype summary for all tables

In [3]:
for name, df in dfs.items():
    print(f"\n{'='*40}")
    print(f"TABLE: {name} → {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(df.dtypes)


TABLE: orders → 99,441 rows × 8 cols
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

TABLE: customers → 99,441 rows × 5 cols
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

TABLE: items → 112,650 rows × 7 cols
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

TABLE: payments → 103,886 rows × 5 cols
order_id                 object
payment_sequential        int64
payment_type            

##  Null audit across all tables

In [4]:
def null_report(name, df):
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"{name}: ✓ no nulls")
        return
    pct = (nulls / len(df) * 100).round(2)
    result = pd.DataFrame({"null_count": nulls, "pct": pct})
    print(f"\n{name}:")
    print(result.to_string())

for name, df in dfs.items():
    null_report(name, df)


orders:
                               null_count   pct
order_approved_at                     160  0.16
order_delivered_carrier_date         1783  1.79
order_delivered_customer_date        2965  2.98
customers: ✓ no nulls
items: ✓ no nulls
payments: ✓ no nulls

reviews:
                        null_count    pct
review_comment_title         87656  88.34
review_comment_message       58247  58.70

products:
                            null_count   pct
product_category_name              610  1.85
product_name_lenght                610  1.85
product_description_lenght         610  1.85
product_photos_qty                 610  1.85
product_weight_g                     2  0.01
product_length_cm                    2  0.01
product_height_cm                    2  0.01
product_width_cm                     2  0.01
sellers: ✓ no nulls
geo: ✓ no nulls
cat_trans: ✓ no nulls


## Duplicate check on primary keys

In [5]:
pk_checks = {
    "orders": ("order_id", orders),
    "customers": ("customer_id", customers),
    "products": ("product_id", products),
    "sellers": ("seller_id", sellers),
}

for name, (pk, df) in pk_checks.items():
    dupes = df[pk].duplicated().sum()
    print(f"{name}.{pk}: {dupes} duplicates")

orders.order_id: 0 duplicates
customers.customer_id: 0 duplicates
products.product_id: 0 duplicates
sellers.seller_id: 0 duplicates


## ## Findings from data audit

**Issues to open on GitHub:**

- [ ] **orders**: ~3% null in `order_delivered_customer_date` (2.98%), ~1.8% null in `order_delivered_carrier_date` (1.79%), ~0.16% null in `order_approved_at` (160 rows)
- [ ] **orders**: all timestamp columns need parsing (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date` are object type → convert to datetime)
- [ ] **products**: ~1.85% null in `product_category_name` (610 rows) — also affects `product_name_lenght`, `product_description_lenght`, `product_photos_qty` (same 610 rows, likely same missing records)
- [ ] **products**: minimal nulls in weight/dimensions (2 rows, ~0.01%)
- [ ] **reviews**: ~58.7% null `review_comment_message` (58,247 rows) — expected for optional field
- [ ] **reviews**: ~88.3% null `review_comment_title` (87,656 rows) — also optional
- [ ] **reviews**: `review_creation_date` and `review_answer_timestamp` need datetime conversion (currently object type)
- [ ] **items**: `shipping_limit_date` needs datetime parsing (currently object type)
- [ ] **geo**: 1,000,163 rows — potential duplicates on `zip_code_prefix` (multiple lat/lng pairs per prefix — check if intentional or needs dedup logic)

**Additional notes (no issue needed):**
- `customers`, `items`, `payments`, `sellers`, `geo`, `cat_trans` — no nulls detected
- No duplicate primary keys found across main tables (`order_id`, `customer_id`, `product_id`, `seller_id`)
- `cat_trans` mapping table is complete (71 rows, no nulls) — ready for category name translation